# Faster R-CNN (ResNet-50 FPN v2)

Download → shared preprocessing/augmentation → training → evaluation for torchvision's `fasterrcnn_resnet50_fpn_v2`. Inputs stay at 640×640 (`min_size=max_size=640`), the same resolution the Ultralytics models get.

In [ ]:
import os

import torch

from ppe_detection_evaluation import FASTER_RCNN
from ppe_detection_evaluation import torchvision_models as tvm
from ppe_detection_evaluation.augment import SharedAugmentation, build_train_transform
from ppe_detection_evaluation.data import check_dataset, class_box_stats, download_dataset, verify_same_examples
from ppe_detection_evaluation.experiment import BATCH_SIZE, DATA_ROOT, EPOCHS, RESULTS_DIR, SEED
from ppe_detection_evaluation.visualize import plot_examples

device = torch.device("cuda" if torch.cuda.is_available() else "mps" if torch.backends.mps.is_available() else "cpu")
device

## 1. Download the dataset

Construction Site Safety **v30** from Roboflow, exported in YOLO TXT format. This release has no augmentations baked
in (unlike the Kaggle mirror, which already contains mosaics), so the shared pipeline below is the only augmentation
any model sees.

Set `ROBOFLOW_API_KEY` in the environment before the first run. `download_dataset` is a no-op once
`data/construction-site-safety-v30/data.yaml` exists, so all four notebooks share one copy.

In [ ]:
download_dataset(DATA_ROOT, api_key=os.environ.get("ROBOFLOW_API_KEY", ""))
summary = check_dataset(DATA_ROOT)  # raises if the class order differs from DEFAULT_CLASS_NAMES
summary

`roboflow_notes` should list only auto-orient / resize to 640×640 and **no augmentation**. The box-size table below is
the gate from AUGMENTATIONS.md §6 Step 0: if more than ~15 % of `Mask` / `NO-Mask` boxes are under 16 px, raise the
minimum of `SCALE_RANGE` from 0.75 to 0.85 in `augment.py` (for every model, not just this one).

In [ ]:
stats = class_box_stats(DATA_ROOT, "train")
for name, s in stats.items():
    print(f"{name:15s} n={s['count']:5d}  <16px {s['under_16px']:6.1%}  <32px {s['under_32px']:6.1%}")

## 2. Preprocessing and augmentation

All four models import the same pipeline from `ppe_detection_evaluation.augment`:

| # | Augmentation | Parameters | p |
|---|---|---|---|
| A1 | Horizontal flip | — | 0.5 |
| A2 | Scale + translate | scale 0.75–1.25, translate ±10 %, gray fill 114 | 1.0 |
| A3 | Colour jitter | brightness ±0.4, contrast ±0.3, saturation ±0.5, hue ±0.015 | 1.0 |
| A4 | One of blur / noise / JPEG | σ 0.1–1.5; σ 0.02–0.06; quality 40–90 | 0.25 |

Boxes keeping less than 40 % of their area, or narrower than 3 px, are dropped. Validation and test images only get
the deterministic preprocessing (load → RGB → 640×640, a no-op for this export).

**Same examples for every model.** `SharedAugmentation` reseeds the pipeline from `(SEED, epoch, image name)` before
each image, so the augmented example is independent of the framework, DataLoader workers, batch size and shuffle
order. With the shared `SEED` and `EPOCHS` from `experiment.py`, every model trains on exactly the same sequence of
examples; only the order within an epoch differs.

In [ ]:
build_train_transform()

The torchvision dataset returns `(image, target)` with the image as a float tensor in [0, 1] and
`target = {"boxes": xyxy pixels, "labels": class_id + 1}` (0 is background, matching `label_offset=1`).
Normalisation and resizing are left to the model's own `GeneralizedRCNNTransform`.

In [ ]:
train_ds = tvm.PPEDataset(DATA_ROOT, "train", SharedAugmentation(seed=SEED))
len(train_ds)

Four training images (columns) as the model sees them in epochs 0–2 (rows):

In [ ]:
examples, titles = [], []
for epoch in range(3):
    train_ds.set_epoch(epoch)
    for i in range(4):
        _, image, boxes, labels = tvm.as_raw_example(train_ds, i)
        examples.append((image, boxes, labels))
        titles.append(f"epoch {epoch} · {train_ds.image_paths[i].name[:20]}")
train_ds.set_epoch(0)
plot_examples(examples, titles);

### Check: this model is fed the shared examples

`verify_same_examples` pulls samples through **this model's own training dataset**, converts them back to RGB uint8
images with pixel boxes, and asserts they equal the output of the shared pipeline. The fingerprints printed here must
be identical in all four model notebooks.

In [ ]:
fingerprints = {}
for epoch in (0, 1):
    train_ds.set_epoch(epoch)
    fingerprints[epoch] = verify_same_examples(
        lambda i: tvm.as_raw_example(train_ds, i), range(min(32, len(train_ds))), train_ds.augmentation, epoch
    )
train_ds.set_epoch(0)
fingerprints

## 3. Model

COCO-pretrained weights with the classification head replaced for 10 classes + background.

In [ ]:
model = tvm.build_faster_rcnn().to(device)
sum(p.numel() for p in model.parameters()) / 1e6  # parameters (M)

## 4. Training

Not run in this notebook. Each epoch calls `train_ds.set_epoch(epoch)` (inside `train_one_epoch`), so the loader
serves that epoch's shared examples. The checkpoint with the best validation mAP@[.5:.95] is kept.

In [ ]:
train_loader = tvm.make_train_loader(train_ds, BATCH_SIZE, SEED)
params = [p for p in model.parameters() if p.requires_grad]
# torchvision detection reference: lr 0.02 at batch 16, scaled linearly to BATCH_SIZE
optimizer = torch.optim.SGD(params, lr=0.02 * BATCH_SIZE / 16, momentum=0.9, weight_decay=1e-4)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=EPOCHS)

ckpt_dir = RESULTS_DIR / "faster_rcnn"
ckpt_dir.mkdir(parents=True, exist_ok=True)
best_map = -1.0
for epoch in range(EPOCHS):
    loss = tvm.train_one_epoch(model, train_loader, optimizer, device, epoch, warmup_iters=500)
    scheduler.step()
    val = tvm.evaluate(model, DATA_ROOT, "valid", FASTER_RCNN, device).evaluate()
    val_map = val["detection"]["groups"]["all"]["mAP"]
    print(f"epoch {epoch}: train loss {loss:.4f}  val mAP@[.5:.95] {val_map:.4f}")
    if val_map > best_map:
        best_map = val_map
        torch.save(model.state_dict(), ckpt_dir / "best.pt")
torch.save(model.state_dict(), ckpt_dir / "last.pt")

## 5. Evaluation

Test images go through the eval transform only. Predictions, ground truth and per-image latency are collected in the
`EvaluationFramework` (see `example.ipynb` for the metric definitions) and saved as JSON for the cross-model comparison.

In [ ]:
model.load_state_dict(torch.load(ckpt_dir / "best.pt", map_location=device))
fw = tvm.evaluate(model, DATA_ROOT, "test", FASTER_RCNN, device)
results = fw.evaluate()
fw.save(RESULTS_DIR / "faster_rcnn_test.json")

In [ ]:
det = results["detection"]["groups"]
print(f"mAP@[.5:.95] {det['all']['mAP']:.4f}   mAP@.5 {det['all']['mAP50']:.4f}")
print(f"PPE mAP@.5 {det['ppe']['mAP50']:.4f}   small PPE mAP@.5 {results['detection']['by_size']['small']['groups']['ppe']['mAP50']:.4f}")
print("violation mean:", results["violation"]["mean"])
print("latency:", results["latency"])